In [1]:
import sdata
import hashlib
from collections import defaultdict
import uuid
import os

In [2]:
namespace = uuid.uuid5(uuid.NAMESPACE_DNS, "sdata")
namespace

UUID('444bc445-2e5c-50c0-b758-d0195dad9867')

In [3]:
def stable_hash_from_list(input_list):
    # Konvertiere die Liste in eine normalisierte Zeichenkette
    normalized_str = str([
        round(item, 8) if isinstance(item, float) else item
        for item in input_list
    ])
    # Erstelle einen Hash mit hashlib (z. B. SHA256)
    hash_object = hashlib.sha256(normalized_str.encode('utf-8'))
    return hash_object.hexdigest()

def stable_uuid_from_list(input_list):
    hashstr = stable_hash_from_list(input_list)
    return generate_uuid_from_hash(hashstr)

In [4]:
class TreeNode:
    def __init__(self, name):
        self.name = name  # Der Hashwert oder Ordnername
        self.children = {}  # Kinderknoten (Unterordner)

    def add_path(self, path):
        if not path:
            return
        head, *tail = path
        if head not in self.children:
            self.children[head] = TreeNode(head)
        self.children[head].add_path(tail)

    def __repr__(self, level=0):
        ret = "  " * level + repr(self.name) + "\n"
        for child in self.children.values():
            ret += child.__repr__(level + 1)
        return ret

def build_tree(hash_lists):
    root = TreeNode("root")
    for hash_list in hash_lists:
        root.add_path(hash_list)
    return root

In [5]:
def create_directories(node, base_path):
    current_path = os.path.join(base_path, node.name)
    if node.name != "root":  # Root nicht erstellen
        os.makedirs(current_path, exist_ok=True)
    for child in node.children.values():
        create_directories(child, current_path)

In [6]:
def generate_uuid_from_hash(hash_value):
    # Konvertiere den Hashwert in einen String (falls nötig)
    hash_string = str(hash_value)
    namespace = uuid.uuid5(uuid.NAMESPACE_DNS, "sdata")
    return uuid.uuid5(namespace, hash_string)

In [7]:

# Beispielnutzung
data = [3.14159265, 42, "hello", 2.7182818]
hashstr = stable_hash_from_list(data)
print(f"Stabiler Hash-Wert: {hashstr}")
stable_uuid_from_list(data)

Stabiler Hash-Wert: f54068f9b31cb94d56b454d36c1458877e5e45589684038b5c60097ad45b9052


UUID('f09e3370-d8f9-5339-8366-753ca7e16eb9')

In [8]:
nested_list = [[3.14159265, 42, "hello", 2.7182818], 
               [1, 1.2,"ssd"],
               ["a"], 
              ]
def stable_hash_from_nested_list(nested_list):

    return [stable_hash_from_list(x) for x in nested_list]
hashlist = stable_hash_from_nested_list(nested_list)
hashlist

['f54068f9b31cb94d56b454d36c1458877e5e45589684038b5c60097ad45b9052',
 'ce6bc57403f6d39b1dd30fe7aca428d5de956758551facfbca290500ad9100b3',
 'a326618a43c0b61aff497f4d8a82f4857fe952c5fae156ab94facf4adbf7dcb4']

In [9]:
import numpy as np
np.array_equal(hashlist, nested_list)

False

In [10]:
generate_uuid_from_hash(hashstr)

UUID('f09e3370-d8f9-5339-8366-753ca7e16eb9')

In [11]:
# Beispiel-Hashwertlisten
hash_lists = [
    ["hash1", "hash2", "hash3"],
    ["hash1", "hash4"],
    ["hash5", "hash6"],
    ["hash1", "hash2", "hash7"]
]

# Baum erstellen
tree = build_tree(hash_lists)
print(tree)

'root'
  'hash1'
    'hash2'
      'hash3'
      'hash7'
    'hash4'
  'hash5'
    'hash6'



In [12]:
base_directory = "/tmp/tree_structure"
create_directories(tree, base_directory)

In [13]:
ls /tmp/tree_structure

root/


In [14]:
uuid.uuid4("ddf")

TypeError: uuid4() takes 0 positional arguments but 1 was given

In [3]:
import hashlib

def cumulative_hash(hashes, new_hash):
    """
    Berechnet einen kumulativen Hash basierend auf einer Liste von vorherigen Hashes
    und einem neuen Hash.
    
    :param hashes: Liste bestehender Hashes (Strings).
    :param new_hash: Neuer Hash (String), der hinzukommt.
    :return: Neuer kumulativer Hash.
    """
    combined = "".join(hashes) + new_hash  # Kombiniere alle bisherigen Hashes mit dem neuen Hash
    return hashlib.sha256(combined.encode('utf-8')).hexdigest()

# Beispielnutzung
hash_list = [
    "f54068f9b31cb94d56b454d36c1458877e5e45589684038b5c60097ad45b9052",
    "ce6bc57403f6d39b1dd30fe7aca428d5de956758551facfbca290500ad9100b3"
]
new_hash = "a326618a43c0b61aff497f4d8a82f4857fe952c5fae156ab94facf4adbf7dcb4"

result = cumulative_hash(hash_list, new_hash)
print(f"Kumulativer Hash: {result}")


Kumulativer Hash: eea7c93318433df0675497c6af463c5c75193b83ac3d612f76b9406e96f409a0


In [2]:
def cumulative_hashes(hashes, new_hashes):
    """
    Berechnet kumulative Hashes iterativ mit einer Liste neuer Hashes.
    
    :param hashes: Liste bestehender Hashes.
    :param new_hashes: Liste neuer Hashes.
    :return: Liste kumulativer Hashes.
    """
    for new_hash in new_hashes:
        cumulative = cumulative_hash(hashes, new_hash)
        hashes.append(cumulative)
    return hashes

hash_list = [
    "f54068f9b31cb94d56b454d36c1458877e5e45589684038b5c60097ad45b9052",
    "ce6bc57403f6d39b1dd30fe7aca428d5de956758551facfbca290500ad9100b3"
]
new_hash = "a326618a43c0b61aff497f4d8a82f4857fe952c5fae156ab94facf4adbf7dcb4"


# Beispielnutzung
new_hashes = [
    "d8e8fca2dc0f896fd7cb4cb0031ba249",
    "45c48cce2e2d7fbdea1afc51c7c6ad26"
]
result = cumulative_hashes(hash_list, new_hashes)
print("Kumulative Hashes:")
for h in result:
    print(h)


NameError: name 'cumulative_hash' is not defined

In [4]:
def generate_combined_hash(parent_hash, own_hash):
    """
    Berechnet einen neuen Hash basierend auf einem Parent-Hash und einem eigenen Hash.
    Falls der Parent-Hash None ist, wird ein leerer String verwendet.
    
    :param parent_hash: Der Hash des Elternknotens (String, None für Root).
    :param own_hash: Der eigene Hash des aktuellen Knotens (String).
    :return: Der kombinierte Hash (String).
    """
    parent_hash = parent_hash or ""  # Falls None, verwende leeren String
    combined = parent_hash + own_hash
    return hashlib.sha256(combined.encode('utf-8')).hexdigest()

In [5]:
# Beispielnutzung
parent_hash = "f54068f9b31cb94d56b454d36c1458877e5e45589684038b5c60097ad45b9052"
own_hash = "a326618a43c0b61aff497f4d8a82f4857fe952c5fae156ab94facf4adbf7dcb4"

result = generate_combined_hash(parent_hash, own_hash)
print(f"Kombinierter Hash: {result}")

Kombinierter Hash: 522be3957dcc77aefb833e2aa65b2d159fab2fe3c52504a1142ffe8c669de604


In [6]:
def cumulative_hash_list(hashes):
    """
    Berechnet kumulative Hashes basierend auf einer Liste, wobei der parent_hash
    jeweils der vorherige Hash in der Liste ist.

    :param hashes: Liste von Hashes (Strings).
    :return: Liste der kumulativ berechneten Hashes.
    """
    if not hashes:
        return []

    cumulative_hashes = [hashes[0]]  # Der erste Hash ist unverändert
    for i in range(1, len(hashes)):
        parent_hash = cumulative_hashes[i - 1]
        current_hash = hashes[i]
        cumulative_hashes.append(generate_combined_hash(parent_hash, current_hash))
    return cumulative_hashes

# Beispielnutzung
hashes = [
    "f54068f9b31cb94d56b454d36c1458877e5e45589684038b5c60097ad45b9052",
    "ce6bc57403f6d39b1dd30fe7aca428d5de956758551facfbca290500ad9100b3",
    "a326618a43c0b61aff497f4d8a82f4857fe952c5fae156ab94facf4adbf7dcb4",
    "b3f1a7f63b85387c26b79521360d1d9306c4a5b6e0333c3b8f2d648b7a6ed8cc",
    "9f86d081884c7d659a2feaa0c55ad015a3bf4f1b2b0b822cd15d6c15b0f00a08"
]

result = cumulative_hash_list(hashes)
print("Kumulative Hashes:")
for i, h in enumerate(result):
    print(f"Hash {i + 1}: {h}")


Kumulative Hashes:
Hash 1: f54068f9b31cb94d56b454d36c1458877e5e45589684038b5c60097ad45b9052
Hash 2: f8c26aec19700d1ecf0714ea4b8581975b74b4f8dbaf06c21fbc890a12b46cff
Hash 3: d445a7bedba5e8b66649752e628f55e2f2dc8e20e91f290ea541b4c7370c47cb
Hash 4: 804329ab9d5b26a28e1af9a701c5c94aa639c6b6813f8928081b21edf00b577c
Hash 5: 82beb5eff82cf82644aeb4e866ebc70e1baf3447fd77de59195012ea97969bc6


In [8]:
hashes = [
    "f54068f9b31cb94d56b454d36c1458877e5e45589684038b5c60097ad45b9052", ]
result = cumulative_hash_list(hashes)
result

['f54068f9b31cb94d56b454d36c1458877e5e45589684038b5c60097ad45b9052']

In [9]:
for i in range(1, len(hashes)):
    print(i)